In [ ]:
import os
import pandas as pd
import numpy as np
import re
import requests

In [ ]:
# function to get unique values
def unique(list1):
 
    # initialize a null list
    unique_list = []
 
    # traverse for all elements
    for x in list1:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
    return unique_list


In [ ]:
# Define user
user = os.getlogin()

# Set file paths
path_sp  = os.path.join('C:\\Users', user, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', user, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config = os.path.join(path_git, 'Python Code', 'DOF', 'config')
print(path_sp)

In [ ]:
# ## User defined functions
# exec(open(os.path.join(path_config, 'Functions.py')).read())

In [ ]:
## Import Variable Mapping
df_inputs = pd.read_excel(os.path.join(path_config, 'DOF Configuration File.xlsx'), sheet_name = 'Inputs', usecols = 'A:E')

# Organize inputs into run
indicator_name = df_inputs['indicator_name'].values[0]
report_theme   = df_inputs['report_theme'  ].values[0]
sp_folder_out  = df_inputs['sp_folder'     ].values[0]
counties       = df_inputs['counties'].dropna().values

print(indicator_name)
print(report_theme  )
print(sp_folder_out )
print(counties      )

***

## Pop_1

***

#### Counties

In [ ]:
## Population Estimates
## E4
# 2020-2024
# 2010-2020
# 2000-2010
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx"
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx"
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx"

## Population and Housing Estimates
## E5
# 2020-2024
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx"
## E8
# 2010-2020
# 2000-2010
url = "https://dof.ca.gov/wp-content/uploads/sites/352/2023/11/E-8_2010_2020_by_Geo_Internet.xlsx"
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx"

In [ ]:
# Household population vs Total population?

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'}

dict_url = {
    2020:"https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx"
    , 2010:"https://dof.ca.gov/wp-content/uploads/sites/352/2023/11/E-8_2010_2020_by_Geo_Internet.xlsx"
    , 2000:"https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx"
}

list_df_state    = []
list_df_counties = []
list_df_cities   = []
list_df_balance  = []

for year in list(dict_url.keys()):
    if year == 2000:
        request = requests.get(dict_url[year], headers = headers)
        request = request.content
        df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
        df = df.dropna(subset = ['Household'])
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        
        # Initialize 'County' and 'City' columns
        df['County'] = np.nan
        df['City'  ] = np.nan
        
        county_temp = None
        
        # Reset index
        df.reset_index(drop=True, inplace=True)
        
        for i in range(len(df)):
            if pd.notnull(df.loc[i, 'County / City']):
                if county_temp is not None:
                    df.loc[i, 'County'] = county_temp
                    df.loc[i, 'City'] = df.loc[i, 'County / City']
                    county_temp = None
                else:
                    county_temp = df.loc[i, 'County / City']
        
        df['County'].fillna(method='ffill', inplace = True)
        df['City'  ].fillna(method='ffill', inplace = True)
        
        # shift up and then fill the last row
        df['County'] = df['County'].shift(-1)
        df['City'  ] = df['City'  ].shift(-1)
        
        df['County'].fillna(method = 'ffill', inplace = True)
        df['City'  ].fillna(method = 'ffill', inplace = True)
        df = df.drop(columns = ['County / City'])
        
        
        df = df[['County', 'City', 'Year', 'Household', 'Total.1']].rename(columns = {'Household':' Household Population', 'Total.1':'Housing Units'})
        df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
        df       = df[df['County'] != 'California']
        
        # Subset
        df_cities   = df[~df['City'].str.contains('County'           )].reset_index(drop = True).drop('County', axis = 1)
        df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
        df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)
        
        list_df_state   .append(df_state   )
        list_df_counties.append(df_counties)
        list_df_cities  .append(df_cities  )
        list_df_balance .append(df_balance )

    else:
        request = requests.get(dict_url[year], headers = headers)
        request = request.content
        df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        
        df = df[['County', 'City', 'Year', 'Household', 'Total.1', 'Occupied']].rename(columns = {'Household':' Household Population', 'Total.1':'Housing Units'})
        df['Unoccupied Housing Units'] = df['Housing Units'] - df['Occupied']
        df = df.drop('Occupied', axis = 1)
        
        df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
        df       = df[df['County'] != 'California']
        
        # Subset
        df_cities   = df[~df['City'].str.contains('County'           )].reset_index(drop = True).drop('County', axis = 1)
        df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
        df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)
        

        list_df_state   .append(df_state   )
        list_df_counties.append(df_counties)
        list_df_cities  .append(df_cities  )
        list_df_balance .append(df_balance )


df_state    = pd.concat(list_df_state   )
df_counties = pd.concat(list_df_counties)
df_cities   = pd.concat(list_df_cities  )
df_balance  = pd.concat(list_df_balance )


df_state    = df_state   .sort_values(['City'  , 'Year'], ascending = [True, False])
df_counties = df_counties.sort_values(['County', 'Year'], ascending = [True, False])
df_cities   = df_cities  .sort_values(['City'  , 'Year'], ascending = [True, False])
df_balance  = df_balance .sort_values(['County', 'Year'], ascending = [True, False])

print('By State')
print(df_state   .head())
print('By Counties')
print(df_counties.head())
print('By Cities')
print(df_cities  .head())
print('By Balance')
print(df_balance .head())

In [ ]:
# i got lucky with finding this user agent on stackoverflow, not sure why it works
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'} 


## Import data
# 2000-2010
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx"
request = requests.get(url, headers = headers)
request = request.content
df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year

df = df[['County', 'City', 'Year', 'Household', 'Total.1', 'Occupied']].rename(columns = {'Household':' Household Population', 'Total.1':'Housing Units'})
df['Unoccupied Housing Units'] = df['Housing Units'] - df['Occupied']
df = df.drop('Occupied', axis = 1)

df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
df       = df[df['County'] != 'California']

# Subset
df_cities   = df[~df['City'].str.contains('County'           )].reset_index(drop = True).drop('County', axis = 1)
df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)


# view
print(df_state   .head())
print(df_cities  .head())
print(df_counties.head())
print(df_balance .head())

df_2020_state    = df_state   .copy()
df_2020_cities   = df_cities  .copy()
df_2020_counties = df_counties.copy()
df_2020_balance  = df_balance .copy()

In [ ]:
# i got lucky with finding this user agent on stackoverflow, not sure why it works
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'} 


## Import data
# 2000-2010
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx"
request = requests.get(url, headers = headers)
request = request.content
df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year

df = df[['County', 'City', 'Year', 'Household', 'Total.1', 'Occupied']].rename(columns = {'Household':' Household Population', 'Total.1':'Housing Units'})
df['Unoccupied Housing Units'] = df['Housing Units'] - df['Occupied']
df = df.drop('Occupied', axis = 1)

df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
df       = df[df['County'] != 'California']

# Subset
df_cities   = df[~df['City'].str.contains('County'           )].reset_index(drop = True).drop('County', axis = 1)
df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)


# view
print(df_state   .head())
print(df_cities  .head())
print(df_counties.head())
print(df_balance .head())

df_2010_state    = df_state   .copy()
df_2010_cities   = df_cities  .copy()
df_2010_counties = df_counties.copy()
df_2010_balance  = df_balance .copy()

In [ ]:
# i got lucky with finding this user agent on stackoverflow, not sure why it works
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'} 

## Import data
# 2000-2010
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx"
request = requests.get(url, headers = headers)
request = request.content
df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
df = df.dropna(subset = ['Household'])
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year

# Initialize 'County' and 'City' columns
df['County'] = np.nan
df['City'  ] = np.nan

county_temp = None

# Reset index
df.reset_index(drop=True, inplace=True)

for i in range(len(df)):
    if pd.notnull(df.loc[i, 'County / City']):
        if county_temp is not None:
            df.loc[i, 'County'] = county_temp
            df.loc[i, 'City'] = df.loc[i, 'County / City']
            county_temp = None
        else:
            county_temp = df_2000.loc[i, 'County / City']

df['County'].fillna(method='ffill', inplace = True)
df['City'  ].fillna(method='ffill', inplace = True)

# shift up and then fill the last row
df['County'] = df['County'].shift(-1)
df['City'  ] = df['City'  ].shift(-1)

df['County'].fillna(method = 'ffill', inplace = True)
df['City'  ].fillna(method = 'ffill', inplace = True)
df = df.drop(columns = ['County / City'])


df = df[['County', 'City', 'Year', 'Household', 'Total.1']].rename(columns = {'Household':' Household Population', 'Total.1':'Housing Units'})
df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
df       = df[df['County'] != 'California']

# Subset
df_cities   = df[~df['City'].str.contains('County'           )].reset_index(drop = True).drop('County', axis = 1)
df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)


# view
print(df_state   .head())
print(df_cities  .head())
print(df_counties.head())
print(df_balance .head())

df_2000_state    = df_state   .copy()
df_2000_cities   = df_cities  .copy()
df_2000_counties = df_counties.copy()
df_2000_balance  = df_balance .copy()

In [ ]:
df_state    = pd.concat([df_2020_state   , df_2010_state   , df_2000_state   ])
df_counties = pd.concat([df_2020_counties, df_2010_counties, df_2000_counties])
df_cities   = pd.concat([df_2020_cities  , df_2010_cities  , df_2000_cities  ])
df_balance  = pd.concat([df_2020_balance , df_2010_balance , df_2000_balance ])

print(df_state   )
print(df_counties)
print(df_cities  )
print(df_balance )

In [ ]:
def restructure_spreadsheet_2000(url, headers):
    dfs = []

    timestamp_matches = re.findall(r'\d+', url)
    if timestamp_matches:
        timestamp = int(timestamp_matches[0])
    else:
        timestamp = None

    df = pd.read_excel(url, sheet_name=1, skiprows=1, header=headers)

    df.iloc[0, :] = df.iloc[0, :].fillna(method='ffill')
    df.iloc[1, :] = df.iloc[1, :].fillna('')

    headers = []
    for category, column in zip(df.iloc[0, :], df.iloc[1, :]):
        if 'Total' in column:
            headers.append((category + ' ' + column).strip())
        else:
            headers.append(column.strip())
    df.columns = headers
    
    df = df.iloc[2:]

    df['Date'] = pd.DatetimeIndex(df['Date']).year
    df = df.dropna(subset=['Date'])
    df['Date'] = df['Date'].astype(int)

    # Initialize 'County' and 'City' columns
    df['County'] = np.nan
    df['City'] = np.nan

    county_temp = None

    # Reset index
    df.reset_index(drop=True, inplace=True)

    for i in range(len(df)):
        if pd.notnull(df.loc[i, 'County / City']):
            if county_temp is not None:
                df.loc[i, 'County'] = county_temp
                df.loc[i, 'City'] = df.loc[i, 'County / City']
                county_temp = None
            else:
                county_temp = df.loc[i, 'County / City']

    df['County'].fillna(method='ffill', inplace=True)
    df['City'].fillna(method='ffill', inplace=True)

    # shift up and then fill the last row
    df['County'] = df['County'].shift(-1)
    df['City'] = df['City'].shift(-1)
    df['County'].fillna(method='ffill', inplace=True)
    df['City'].fillna(method='ffill', inplace=True)

    df = df.drop(columns=['County / City'])

    # Create 'Census Benchmark' column
    df['Census Benchmark'] = np.where(df['Date'].eq(df['Date'].shift()), 'Yes', 'No')

    df = df[['Date', 'County', 'City', 'Census Benchmark'] + [col for col in df.columns if col not in ['Date', 'City', 'County', 'Census Benchmark']]]

    df['SF'] = df['Single'] + df['Mobile Homes']
    df['MF'] = df['Multiple']
    df.drop('Multiple', axis = 1, inplace = True)

    return df

In [ ]:
unique(df_2000w[df_2000w['City'].str.contains('County')].City.values)

In [ ]:
# ## Indicator Pop_1 Datasets (E4)

# if indicator_name == 'Pop_1':
    
#     ## Outline
#     # Set fake user agent to avoid 403 error
#     # Set URL of table, starting here https://dof.ca.gov/forecasting/demographics/
#     # Request import of excel workbook with URL
#     # Convert request content to pandas dataframe
#     # Drop missings, remove state totals
#     # Reshape data to long format, clean date field, reshape back to wide
#     # Clean column names
#     # Repeat for data from 2000-2010, 2010-2020, 2020-2024
#     # Stack all data together
#     # Subset to counties as needed
    
    
#     # i got lucky with finding this user agent on stackoverflow, not sure why it works
#     headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'} 
    
    
#     ## Import data
#     # 2000-2010
#     url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx"
#     request = requests.get(url, headers = headers)
#     request = request.content
#     df_2000w = pd.read_excel(request, sheet_name = 1, skiprows = 3, engine='openpyxl')
#     df_2000w = df_2000w.dropna()
#     df_2000w = df_2000w[df_2000w['COUNTY'] != 'State Total']
#     df_2000 = pd.melt(df_2000w
#                         , id_vars = ['COUNTY']
#                         , var_name = 'Date'
#                         , value_name = 'Total'
#                      )
#     df_2000['Date'] = pd.to_datetime(df_2000['Date'])
#     df_2000w = df_2000.pivot_table(index = ['COUNTY']
#                                        , columns = 'Date'
#                                        , values = 'Total').reset_index()
#     df_2000w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2000w.columns]
    
#     # 2010-2020
#     url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx"
#     request = requests.get(url, headers = headers)
#     request = request.content
#     df_2010w = pd.read_excel(request, sheet_name = 1, skiprows = 1, engine='openpyxl')
#     df_2010w = df_2010w[df_2010w['COUNTY'] != 'State Total']
#     df_2010w = df_2010w.dropna()
#     df_2010 = pd.melt(df_2010w
#                         , id_vars = ['COUNTY']
#                         , var_name = 'Date'
#                         , value_name = 'Total'
#                      )
#     df_2010['Date'] = pd.to_datetime(df_2010['Date'])
#     df_2010w = df_2010.pivot_table(index = ['COUNTY']
#                                        , columns = 'Date'
#                                        , values = 'Total').reset_index()
#     df_2010w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2010w.columns]
    
#     # 2020-2024
#     url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx"
#     request = requests.get(url, headers = headers)
#     request = request.content
#     df_2020w = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
#     df_2020w = df_2020w[df_2020w['COUNTY'] != 'State Total']
#     df_2020w = df_2020w.dropna()
#     df_2020 = pd.melt(df_2020w
#                         , id_vars = ['COUNTY']
#                         , var_name = 'Date'
#                         , value_name = 'Total'
#                      )
#     df_2020['Date'] = pd.to_datetime(df_2020['Date'])
#     df_2020w = df_2020.pivot_table(index = ['COUNTY']
#                                        , columns = 'Date'
#                                        , values = 'Total').reset_index()
#     df_2020w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2020w.columns]
    
#     ## Combine data
    
#     # wide
#     df_dof2 = df_2000w.merge(df_2010w, on = 'COUNTY', how = 'left')
#     df_dof2 = df_dof2 .merge(df_2020w, on = 'COUNTY', how = 'left')
    
#     # long
#     df_dof = pd.concat([df_2000, df_2010, df_2020])
#     df_dof = df_dof.sort_values(['COUNTY', 'Date'])

#     # Clean COUNTY field
#     df_dof ['COUNTY'] = df_dof ['COUNTY'].apply(lambda s : re.sub('[\s+]', ' ', s.strip()))
#     df_dof2['COUNTY'] = df_dof2['COUNTY'].apply(lambda s : re.sub('[\s+]', ' ', s.strip()))

#     ## Subset to counties as needed
#     df_dof  = df_dof [df_dof ['COUNTY'].isin(counties)]
#     df_dof2 = df_dof2[df_dof2['COUNTY'].isin(counties)]

# print('Success!  DOF data imported correctly probably')

In [ ]:
# df_dof.head()

In [ ]:
# df_dof2.head()

In [ ]:
# Export to SP
# set(df_dof['COUNTY/CITY'].values)
# Need to double check if:
# These are actually jurisdictions
# do we want "Incorporated" or other weird assignments

#### Jurisdictions

In [ ]:
# if indicator_name == 'Pop_1':
    
#     ## Outline
#     # Set fake user agent to avoid 403 error
#     # Set URL of table, starting here https://dof.ca.gov/forecasting/demographics/
#     # Request import of excel workbook with URL
#     # Convert request content to pandas dataframe
#     # Drop missings, remove state totals
#     # Reshape data to long format, clean date field, reshape back to wide
#     # Clean column names
#     # Repeat for data from 2000-2010, 2010-2020, 2020-2024
#     # Stack all data together
#     # Subset to counties as needed
    
    
#     # i got lucky with finding this user agent on stackoverflow, not sure why it works
#     headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'} 
    
    
#     ## Import data
#     # 2000-2010
#     url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx"
#     request = requests.get(url, headers = headers)
#     request = request.content
#     df_2000w = pd.read_excel(request, sheet_name = 2, skiprows = 3, engine='openpyxl')
#     df_2000w = df_2000w.dropna()
#     df_2000w = df_2000w[~df_2000w['COUNTY/CITY'].str.contains('Total')].reset_index(drop = True)
#     df_2000 = pd.melt(df_2000w
#                         , id_vars = ['COUNTY/CITY']
#                         , var_name = 'Date'
#                         , value_name = 'Total'
#                      )
#     df_2000['Date'] = pd.to_datetime(df_2000['Date'])
#     df_2000w = df_2000.pivot_table(index = ['COUNTY/CITY']
#                                        , columns = 'Date'
#                                        , values = 'Total').reset_index()
#     df_2000w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2000w.columns]

#     # 2010-2020
#     url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx"
#     request = requests.get(url, headers = headers)
#     request = request.content
#     df_2010w = pd.read_excel(request, sheet_name = 2, skiprows = 1, engine='openpyxl', usecols = 'A:L')
#     df_2010w = df_2010w.dropna()
#     df_2010w = df_2010w[~df_2010w['COUNTY/CITY'].str.contains('Total')].reset_index(drop = True)
#     df_2010 = pd.melt(df_2010w
#                         , id_vars = ['COUNTY/CITY']
#                         , var_name = 'Date'
#                         , value_name = 'Total'
#                      )
#     df_2010['Date'] = pd.to_datetime(df_2010['Date'])
#     df_2010w = df_2010.pivot_table(index = ['COUNTY/CITY']
#                                        , columns = 'Date'
#                                        , values = 'Total').reset_index()
#     df_2010w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2010w.columns]
    
#     # 2020-2024
#     url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx"
#     request = requests.get(url, headers = headers)
#     request = request.content
#     df_2020w = pd.read_excel(request, sheet_name = 2, skiprows = 2, engine='openpyxl', usecols = 'A:F')
#     df_2020w = df_2020w.dropna()
#     df_2020w = df_2020w[~df_2020w['COUNTY/CITY'].str.contains('Total')].reset_index(drop = True)
#     df_2020 = pd.melt(df_2020w
#                         , id_vars = ['COUNTY/CITY']
#                         , var_name = 'Date'
#                         , value_name = 'Total'
#                      )
#     df_2020['Date'] = pd.to_datetime(df_2020['Date'])
#     df_2020w = df_2020.pivot_table(index = ['COUNTY/CITY']
#                                        , columns = 'Date'
#                                        , values = 'Total').reset_index()
#     df_2020w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2020w.columns]
    
    
#     ## Combine data
    
#     # wide
#     df_dof2 = df_2000w.merge(df_2010w, on = 'COUNTY/CITY', how = 'left')
#     df_dof2 = df_dof2 .merge(df_2020w, on = 'COUNTY/CITY', how = 'left')
    
#     # long
#     df_dof = pd.concat([df_2000, df_2010, df_2020])
#     df_dof = df_dof.sort_values(['COUNTY/CITY', 'Date'])

#     # Clean COUNTY field
#     df_dof ['COUNTY/CITY'] = df_dof ['COUNTY/CITY'].apply(lambda s : re.sub('[\s+]', ' ', s.strip()))
#     df_dof2['COUNTY/CITY'] = df_dof2['COUNTY/CITY'].apply(lambda s : re.sub('[\s+]', ' ', s.strip()))

# print('Success!  DOF data imported correctly probably')

***

## Cost_3

***

Code graveyard

In [ ]:
# header = {
#   "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
#   "X-Requested-With": "XMLHttpRequest"
# }

# r = requests.get(site, headers=header)
# r.text

In [ ]:
# import pandas as pd
# import requests

# # Check the end of the url -->                                                                             HERE --v
# url = 'https://<myOrg>.sharepoint.com/:x:/s/x-taulukot/Ec0R1y3l7sdGsP92csSO-mgBI8WCN153LfEMvzKMSg1Zzg?e=6NS5Qh&download=1'
# headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'}

# resp = requests.get(url, headers=headers)
# df = pd.read_excel(resp.content, engine='openpyxl')